CS 4120 Final Project

Ethan Tang

Data Cleaning and Feature Engineering

Book reviews labeled with spoiler (0/1) at the review level.

https://cseweb.ucsd.edu/~jmcauley/datasets/goodreads.html

### Data Cleaning

In [1]:
import pandas as pd

df = pd.read_json("goodreads_reviews_spoiler_raw.json.gz", lines=True)
df

,user_id,book_id,review_id,rating,review_text,date_added,date_updated,read_at,started_at,n_votes,n_comments
0,8842281e1d1347389f2ab93d60773d4d,18245960,dfdbb7b0eb5a7e4c26d59a937e2e5feb,5,This is a special book. It started slow for ab...,Sun Jul 30 07:44:10 -0700 2017,Wed Aug 30 00:00:26 -0700 2017,Sat Aug 26 12:05:52 -0700 2017,Tue Aug 15 13:23:18 -0700 2017,28,1
1,8842281e1d1347389f2ab93d60773d4d,16981,a5d2c3628987712d0e05c4f90798eb67,3,Recommended by Don Katz. Avail for free in Dec...,Mon Dec 05 10:46:44 -0800 2016,Wed Mar 22 11:37:04 -0700 2017,,,1,0
2,8842281e1d1347389f2ab93d60773d4d,28684704,2ede853b14dc4583f96cf5d120af636f,3,"A fun, fast paced science fiction thriller. I ...",Tue Nov 15 11:29:22 -0800 2016,Mon Mar 20 23:40:27 -0700 2017,Sat Mar 18 23:22:42 -0700 2017,Fri Mar 17 23:45:40 -0700 2017,22,0
3,8842281e1d1347389f2ab93d60773d4d,27161156,ced5675e55cd9d38a524743f5c40996e,0,Recommended reading to understand what is goin...,Wed Nov 09 17:37:04 -0800 2016,Wed Nov 09 17:38:20 -0800 2016,,,5,1
4,8842281e1d1347389f2ab93d60773d4d,25884323,332732725863131279a8e345b63ac33e,4,"I really enjoyed this book, and there is a lot...",Mon Apr 25 09:31:23 -0700 2016,Mon Apr 25 09:31:23 -0700 2016,Sun Jun 26 00:00:00 -0700 2016,Sat May 28 00:00:00 -0700 2016,9,1
...,...,...,...,...,...,...,...,...,...,...,...
1378028,35cef391b171b4fca45771e508028212,15745950,0e1db3d4b04256f9660f5d276ddf1314,0,Can't wait for Travis' POV \n Travis Before Ab...,Sun Aug 05 10:26:12 -0700 2012,Tue Apr 16 17:24:00 -0700 2013,Tue Apr 16 00:00:00 -0700 2013,,0,0
1378029,35cef391b171b4fca45771e508028212,10861195,0b7f352e58caf0fd1f961e98ef04e89c,0,Had this on my to-read shelf forever. Will upd...,Tue Jul 10 23:31:00 -0700 2012,Fri Dec 28 20:05:51 -0800 2012,,,0,0
1378030,35cef391b171b4fca45771e508028212,6131164,9b19eff33ddb14e9e68fca2e90379e46,4,The last book left me wanting for more. I need...,Tue Jul 10 19:45:17 -0700 2012,Mon Mar 25 18:41:51 -0700 2013,Tue Mar 19 00:00:00 -0700 2013,,0,0
1378031,35cef391b171b4fca45771e508028212,10025305,8be463fed78f0da63e964706f710332b,4,Things are heating up in the second novel of I...,Thu Jul 05 19:19:30 -0700 2012,Thu Jan 24 16:24:54 -0800 2013,Mon Jan 14 00:00:00 -0800 2013,,0,0


In [2]:
df = df[['book_id', 'review_text']].copy()
df

,book_id,review_text
0,18245960,This is a special book. It started slow for ab...
1,16981,Recommended by Don Katz. Avail for free in Dec...
2,28684704,"A fun, fast paced science fiction thriller. I ..."
3,27161156,Recommended reading to understand what is goin...
4,25884323,"I really enjoyed this book, and there is a lot..."
...,...,...
1378028,15745950,Can't wait for Travis' POV \n Travis Before Ab...
1378029,10861195,Had this on my to-read shelf forever. Will upd...
1378030,6131164,The last book left me wanting for more. I need...
1378031,10025305,Things are heating up in the second novel of I...


In [3]:
# Create binary feature for if a review contains a spoiler.

import re

# Create the binary spoiler label from the official Goodreads spoiler markup before any text is changed.
df['has_spoiler'] = df['review_text'].str.contains('(view spoiler)[', regex=False, na=False).astype(int)

# Compile a case-insensitive pattern that matches the official spoiler markers as well as malformed variants such as "View Spoiler >>".
SPOILER_MARKER_RE = re.compile(r'\(?\b(?:view|hide) spoiler\b\)?\s*(?:\[|\]|>>)?', flags=re.IGNORECASE)

# Flag the reviews that contain the word "spoiler" so that the slower regex steps run only on those rows.
mentions_spoiler = df['review_text'].str.contains('spoiler', case=False, regex=False, na=False)

# Replace every spoiler marker with a space while keeping the hidden spoiler text itself.
df.loc[mentions_spoiler, 'review_text'] = df.loc[mentions_spoiler, 'review_text'].str.replace(SPOILER_MARKER_RE, ' ', regex=True)

# Confirm that no marker remains, since a leftover marker would leak the label to the model.
assert not df.loc[mentions_spoiler, 'review_text'].str.contains(SPOILER_MARKER_RE).any()

df

,book_id,review_text,has_spoiler
0,18245960,This is a special book. It started slow for ab...,1
1,16981,Recommended by Don Katz. Avail for free in Dec...,0
2,28684704,"A fun, fast paced science fiction thriller. I ...",1
3,27161156,Recommended reading to understand what is goin...,0
4,25884323,"I really enjoyed this book, and there is a lot...",1
...,...,...,...
1378028,15745950,Can't wait for Travis' POV \n Travis Before Ab...,0
1378029,10861195,Had this on my to-read shelf forever. Will upd...,0
1378030,6131164,The last book left me wanting for more. I need...,0
1378031,10025305,Things are heating up in the second novel of I...,0


In [5]:
# Normalize the text.

# Apply NFKC Unicode normalization so that visually identical characters share one encoding.
df['review_text'] = df['review_text'].str.normalize('NFKC')

# Compile a pattern for genuine HTML formatting tags only, because most angle-bracket text in these reviews is written by the reviewers themselves.
HTML_TAG_RE = re.compile(r'</?(?:i|b|u|em|strong|br|p|div|span|blockquote)\s*/?>|<a\s+href=[^>]*>|</a>', flags=re.IGNORECASE)

# Replace each HTML tag with a space.
df['review_text'] = df['review_text'].str.replace(HTML_TAG_RE, ' ', regex=True)

# Compile a case-insensitive pattern for web addresses.
URL_RE = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)

# Flag the reviews that may contain a web address so that the regex replacement runs only on those rows.
has_url = df['review_text'].str.contains('http', case=False, regex=False, na=False) | df['review_text'].str.contains('www.', case=False, regex=False, na=False)

# Replace each web address with a single placeholder token.
df.loc[has_url, 'review_text'] = df.loc[has_url, 'review_text'].str.replace(URL_RE, ' URL ', regex=True)

# Split the text on any run of whitespace and rejoins it with single spaces, which also removes newlines and trims both ends.
df['review_text'] = df['review_text'].str.split().str.join(' ')

In [6]:
# Remove missing and short reviews.

# Record the row count before filtering.
n_before = len(df)

# Drop rows whose review text is missing.
df = df.dropna(subset=['review_text'])

# Set the minimum number of words that a review needs in order to carry useful signal.
MIN_WORDS = 5

# Count the whitespace-separated words in each cleaned review.
word_counts = df['review_text'].str.count(r'\S+')

# Keep only the reviews that meet the minimum word count, which also removes empty reviews.
df = df[word_counts >= MIN_WORDS]

# Report how many rows were removed and how many remain.
print(f'Removed {n_before - len(df):,} missing or short reviews; {len(df):,} remain.')

Removed 25,866 missing or short reviews; 1,352,167 remain.


In [7]:
# Remove duplicates.

# Record the row count before deduplication.
n_before = len(df)

# Remove repeated copies of the same text with the same label, keeping the first occurrence.
df = df.drop_duplicates(subset=['review_text', 'has_spoiler'], keep='first')

# Remove every copy of any text that still appears more than once, since those copies carry conflicting labels.
df = df[~df.duplicated(subset=['review_text'], keep=False)]

# Report how many rows were removed and how many remain.
print(f'Removed {n_before - len(df):,} duplicate reviews; {len(df):,} remain.')

df

Removed 7,450 duplicate reviews; 1,344,717 remain.


,book_id,review_text,has_spoiler
0,18245960,This is a special book. It started slow for ab...,1
1,16981,Recommended by Don Katz. Avail for free in Dec...,0
2,28684704,"A fun, fast paced science fiction thriller. I ...",1
3,27161156,Recommended reading to understand what is goin...,0
4,25884323,"I really enjoyed this book, and there is a lot...",1
...,...,...,...
1378028,15745950,Can't wait for Travis' POV Travis Before Abby....,0
1378029,10861195,Had this on my to-read shelf forever. Will upd...,0
1378030,6131164,The last book left me wanting for more. I need...,0
1378031,10025305,Things are heating up in the second novel of I...,0


In [10]:
# Print the total number of reviews that contain a spoiler.
df['has_spoiler'].sum()

89586

In [ ]:
df.to_csv('df_book_reviews_cleaned.csv', index=False)

### Feature Engineering for Logistic Regression

In [8]:
df = pd.read_csv('df_book_reviews_cleaned.csv')

# Set up the feature engineering steps.

# Import the os module to read the number of CPU cores.
import os

# Import NumPy for numerical arrays.
import numpy as np

# Import SciPy's sparse matrix module for the word-count and TF-IDF matrices.
import scipy.sparse as sp

# Import joblib for parallel processing and for saving the fitted vectorizer.
from joblib import Parallel, delayed, dump

# Import the scikit-learn text vectorizers.
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer

# Use at most eight processes so that the parallel steps stay within memory limits.
N_JOBS = min(8, os.cpu_count())

# Define a helper that applies a fixed-vocabulary vectorizer to the reviews in parallel chunks.
def transform_in_parallel(vectorizer, texts):
    # Split the texts into four chunks per process so that the work stays evenly balanced.
    chunks = np.array_split(texts.to_numpy(), N_JOBS * 4)
    # Transform every chunk in a separate process.
    parts = Parallel(n_jobs=N_JOBS)(delayed(vectorizer.transform)(chunk) for chunk in chunks)
    # Stack the chunk matrices back together in the original row order.
    return sp.vstack(parts, format='csr')

In [9]:
# Feature Type 1: Compute TF-IDF weights for word unigrams and bigrams.

# Set the number of reviews used to choose the vocabulary, because a bigram vocabulary built from every review would need too much memory.
VOCAB_SAMPLE_SIZE = 200_000

# Configure a counter that keeps the 300,000 most frequent unigrams and bigrams that appear in at least 5 sampled reviews and in at most 90% of them.
vocab_counter = CountVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, max_features=300_000)

# Learn the vocabulary from a reproducible random sample of reviews.
vocab_counter.fit(df['review_text'].sample(VOCAB_SAMPLE_SIZE, random_state=42))

# Keep only the learned vocabulary.
vocabulary = vocab_counter.vocabulary_

# Free the memory that the counter uses to store every discarded term.
del vocab_counter

# Configure a counter that counts only the learned vocabulary terms.
term_counter = CountVectorizer(ngram_range=(1, 2), vocabulary=vocabulary, dtype=np.float32)

# Count the vocabulary terms in every review in parallel.
term_counts = transform_in_parallel(term_counter, df['review_text'])

# Configure a transformer with sublinear term frequency, so that a word repeated many times does not dominate a review.
tfidf_transformer = TfidfTransformer(sublinear_tf=True)

# Learn the inverse document frequencies from all reviews.
tfidf_transformer.fit(term_counts)

# Convert the counts into a TF-IDF matrix with one row per row of df, without making an extra copy of the counts.
X_tfidf = tfidf_transformer.transform(term_counts, copy=False)

# Free the memory used by the raw counts.
del term_counts

# Build a TF-IDF vectorizer that converts new raw text into the same features.
tfidf = TfidfVectorizer(ngram_range=(1, 2), vocabulary=vocabulary, sublinear_tf=True, dtype=np.float32)

# Copy the learned inverse document frequencies into the vectorizer.
tfidf.idf_ = tfidf_transformer.idf_

# Save the TF-IDF matrix to disk in compressed form.
sp.save_npz('book_X_tfidf.npz', X_tfidf, compressed=True)

# Save the fitted vectorizer to disk.
dump(tfidf, 'book_tfidf_vectorizer.joblib')

# Display the shape of the TF-IDF matrix.
X_tfidf.shape

(1344717, 300000)

In [ ]:
# Feature Type 2: Measure how specific each review's words are to the reviewed book.

# Mark which vocabulary terms are single words, because book-specific terms such as character names are mostly unigrams.
is_unigram = np.array([' ' not in term for term in tfidf.get_feature_names_out()])

# Build a review-by-word matrix that records which vocabulary words appear in each review.
word_presence = X_tfidf[:, is_unigram]

# Replace the TF-IDF weights with ones so that the matrix records presence only.
word_presence.data[:] = 1

# Encode each book as an integer from 0 to the number of books minus 1.
book_codes, book_ids = pd.factorize(df['book_id'])

# Store the number of books, reviews, and vocabulary words.
n_books, n_reviews, n_words = len(book_ids), word_presence.shape[0], word_presence.shape[1]

# Build a book-by-review indicator matrix that links every review to its book.
book_reviews = sp.csr_matrix((np.ones(n_reviews, dtype=np.float32), (book_codes, np.arange(n_reviews))), shape=(n_books, n_reviews))

# Count, for every book and word, how many of the book's reviews contain the word.
book_word_counts = (book_reviews @ word_presence).tocsr()

# Sort the word indices within each book so that the book-word pairs are in ascending order.
book_word_counts.sort_indices()

# Count the reviews of each book.
reviews_per_book = np.bincount(book_codes, minlength=n_books)

# Count how many books have at least one review that contains each word.
books_per_word = np.bincount(book_word_counts.indices, minlength=n_words)

# Compute the inverse item frequency of each word, which is high for words that appear in the reviews of only a few books.
inverse_item_freq = np.log(n_books / np.maximum(books_per_word, 1)).astype(np.float32)

# Give every book-word pair a unique integer key, which is in ascending order because the pairs are sorted.
pair_keys = np.repeat(np.arange(n_books, dtype=np.int64), np.diff(book_word_counts.indptr)) * n_words + book_word_counts.indices

# Allocate an array for the mean specificity score of each review.
specificity_mean = np.zeros(n_reviews, dtype=np.float32)

# Allocate an array for the maximum specificity score of each review.
specificity_max = np.zeros(n_reviews, dtype=np.float32)

# Set the number of reviews processed at once, which limits memory use.
CHUNK_SIZE = 200_000

# Process the reviews one chunk at a time.
for start in range(0, n_reviews, CHUNK_SIZE):
    # Select the word-presence rows of the current chunk.
    chunk = word_presence[start:start + CHUNK_SIZE]
    # Find the book of the review behind every word occurrence in the chunk.
    occurrence_books = book_codes[start + np.repeat(np.arange(chunk.shape[0]), np.diff(chunk.indptr))]
    # Look up how many reviews of the same book contain each word.
    book_counts = book_word_counts.data[np.searchsorted(pair_keys, occurrence_books * n_words + chunk.indices)]
    # Count the other reviews of the same book, which excludes the review itself.
    other_reviews = reviews_per_book[occurrence_books] - 1
    # Compute the share of the book's other reviews that contain each word, using zero for books with a single review.
    other_review_share = np.divide(book_counts - 1, other_reviews, out=np.zeros(len(book_counts), dtype=np.float32), where=other_reviews > 0)
    # Multiply the share by the inverse item frequency to score how specific each word is to the book.
    scores = sp.csr_matrix((other_review_share * inverse_item_freq[chunk.indices], chunk.indices, chunk.indptr), shape=chunk.shape)
    # Average the word scores within each review.
    specificity_mean[start:start + CHUNK_SIZE] = scores.sum(axis=1).A1 / np.maximum(np.diff(chunk.indptr), 1)
    # Keep the highest word score within each review.
    specificity_max[start:start + CHUNK_SIZE] = scores.max(axis=1).toarray().ravel()

# Add the mean specificity score to df.
df['book_specificity_mean'] = specificity_mean

# Add the maximum specificity score to df.
df['book_specificity_max'] = specificity_max

# Free the memory used by the intermediate arrays.
del word_presence, book_reviews, book_word_counts, pair_keys, chunk, occurrence_books, book_counts, other_reviews, other_review_share, scores

# Display summary statistics of the new features.
df[['book_specificity_mean', 'book_specificity_max']].describe()

,book_specificity_mean,book_specificity_max
count,1.344717e+06,1.344717e+06
mean,6.801393e-02,1.843975e+00
std,6.930344e-02,1.392552e+00
min,0.000000e+00,0.000000e+00
25%,2.674329e-02,4.596093e-01
50%,5.453805e-02,1.846613e+00
75%,8.625420e-02,2.824922e+00
max,3.325164e+00,8.400638e+00


In [ ]:
# Feature Type 3: Compute structural and stylistic features from the review text.

# Count the words in each review, since cleaning left exactly one space between consecutive words.
word_count = df['review_text'].str.count(' ') + 1

# Count the sentences in each review as runs of end punctuation, and treat every review as having at least one sentence.
sentence_count = df['review_text'].str.count(r'[.!?]+(?:\s|$)').clip(lower=1)

# Count the capitalized words that follow a lowercase letter, comma, colon, or semicolon, which mostly captures names in the middle of a sentence.
mid_sentence_capitals = df['review_text'].str.count(r'[a-z,;:] (?=[A-Z][a-z])')

# List the first-person pronouns, which signal that the reviewer is giving an opinion.
FIRST_PERSON = ['i', 'me', 'my', 'mine', 'myself', 'we', 'us', 'our', 'ours', 'ourselves']

# List the third-person pronouns, which signal that the reviewer is retelling the plot.
THIRD_PERSON = ['he', 'him', 'his', 'himself', 'she', 'her', 'hers', 'herself', 'they', 'them', 'their', 'theirs', 'themselves']

# Configure a counter for the pronouns that also keeps one-letter words such as "i".
pronoun_counter = CountVectorizer(vocabulary=FIRST_PERSON + THIRD_PERSON, token_pattern=r'(?u)\b\w+\b', dtype=np.float32)

# Count the pronouns in every review in parallel.
pronoun_counts = transform_in_parallel(pronoun_counter, df['review_text'])

# Add up the first-person pronouns in each review.
first_person_count = pronoun_counts[:, :len(FIRST_PERSON)].sum(axis=1).A1

# Add up the third-person pronouns in each review.
third_person_count = pronoun_counts[:, len(FIRST_PERSON):].sum(axis=1).A1

# Add the log-scaled word count to df, because review lengths are heavily skewed.
df['log_word_count'] = np.log1p(word_count).astype(np.float32)

# Add the log-scaled sentence count to df.
df['log_sentence_count'] = np.log1p(sentence_count).astype(np.float32)

# Add the log-scaled average number of words per sentence to df.
df['log_avg_sentence_length'] = np.log1p(word_count / sentence_count).astype(np.float32)

# Add the share of words that are capitalized in the middle of a sentence to df.
df['capitalized_word_density'] = (mid_sentence_capitals / word_count).astype(np.float32)

# Add the log ratio of third-person to first-person pronouns to df, adding one to each count to avoid division by zero.
df['log_third_first_pronoun_ratio'] = np.log((third_person_count + 1) / (first_person_count + 1)).astype(np.float32)

# Add the log-scaled count of quotation marks to df.
df['log_quote_count'] = np.log1p(df['review_text'].str.count('"')).astype(np.float32)

# Add the log-scaled count of exclamation marks to df.
df['log_exclamation_count'] = np.log1p(df['review_text'].str.count('!')).astype(np.float32)

# Add the log-scaled count of question marks to df.
df['log_question_count'] = np.log1p(df['review_text'].str.count(r'\?')).astype(np.float32)

# Add the log-scaled count of numbers to df.
df['log_number_count'] = np.log1p(df['review_text'].str.count(r'\d+')).astype(np.float32)

# Free the memory used by the intermediate results.
del word_count, sentence_count, mid_sentence_capitals, pronoun_counts, first_person_count, third_person_count

# Display summary statistics of the new features.
df.loc[:, 'log_word_count':].describe()

,log_word_count,log_sentence_count,log_avg_sentence_length,capitalized_word_density,log_third_first_pronoun_ratio,log_quote_count,log_exclamation_count,log_question_count,log_number_count
count,1.344717e+06,1.344717e+06,1.344717e+06,1.344717e+06,1.344717e+06,1.344717e+06,1.344717e+06,1.344717e+06,1.344717e+06
mean,4.657557e+00,2.172880e+00,2.734115e+00,3.917796e-02,-5.904008e-01,3.771488e-01,4.420344e-01,2.929781e-01,4.681940e-01
std,1.221004e+00,9.076247e-01,4.220876e-01,4.553988e-02,9.482237e-01,7.165703e-01,6.704281e-01,5.543768e-01,6.500823e-01
min,1.791759e+00,6.931472e-01,6.931472e-01,0.000000e+00,-4.290460e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.784190e+00,1.386294e+00,2.505526e+00,5.434783e-03,-1.252763e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,4.744932e+00,2.079442e+00,2.772589e+00,3.061225e-02,-6.931472e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,5.609472e+00,2.833213e+00,3.001963e+00,5.504587e-02,0.000000e+00,0.000000e+00,6.931472e-01,6.931472e-01,1.098612e+00
max,8.158802e+00,5.117994e+00,6.159095e+00,9.523810e-01,4.983607e+00,5.187386e+00,5.365976e+00,4.691348e+00,5.638355e+00


In [ ]:
# Feature Type 4: spoiler keyword count and spoiler keyword flag.

# List cue words about death.
DEATH_CUES = ['die', 'dies', 'died', 'dying', 'death', 'dead', 'kill', 'kills', 'killed', 'killing', 'killer', 'murder', 'murdered', 'murderer']

# List cue words and phrases about endings.
ENDING_CUES = ['ending', 'ended', 'ends up', 'ended up', 'the end', 'epilogue', 'finale', 'last chapter', 'final chapter', 'last page']

# List cue words and phrases about twists and revelations.
TWIST_CUES = ['twist', 'twists', 'reveal', 'reveals', 'revealed', 'revelation', 'turns out', 'turned out', 'finds out', 'found out', 'cliffhanger', 'betray', 'betrays', 'betrayed', 'betrayal']

# List cue words about major life events of the characters.
LIFE_EVENT_CUES = ['married', 'marries', 'marry', 'wedding', 'pregnant', 'pregnancy', 'survive', 'survives', 'survived', 'sacrifice', 'sacrificed']

# List the words that mention spoilers directly.
SPOILER_MENTIONS = ['spoiler', 'spoilers']

# List the phrases that state that a review avoids spoilers, which are subtracted from the count.
NEGATED_MENTIONS = ['spoiler free', 'spoilers free', 'no spoiler', 'no spoilers']

# Combine the cue lists into one list of keywords.
SPOILER_KEYWORDS = DEATH_CUES + ENDING_CUES + TWIST_CUES + LIFE_EVENT_CUES + SPOILER_MENTIONS

# Configure a counter for the keywords and the negated phrases.
keyword_counter = CountVectorizer(ngram_range=(1, 2), vocabulary=SPOILER_KEYWORDS + NEGATED_MENTIONS, dtype=np.float32)

# Count the keywords and negated phrases in every review in parallel.
keyword_counts = transform_in_parallel(keyword_counter, df['review_text'])

# Add up the keyword occurrences in each review.
cue_count = keyword_counts[:, :len(SPOILER_KEYWORDS)].sum(axis=1).A1

# Add up the negated phrases in each review.
negated_count = keyword_counts[:, len(SPOILER_KEYWORDS):].sum(axis=1).A1

# Detect the "** spoiler alert **" header that Goodreads adds to the start of some reviews, because the reviewer did not write it.
has_alert_header = df['review_text'].str.match(r'\*\* spoiler alert \*\*', case=False).to_numpy()

# Add the keyword count to df, removing one mention of "spoiler" for each negated phrase and for the header, and never going below zero.
df['spoiler_keyword_count'] = np.clip(cue_count - negated_count - has_alert_header, 0, None).astype(np.float32)

# Add a flag to df that marks whether a review contains at least one keyword.
df['spoiler_keyword_flag'] = (df['spoiler_keyword_count'] > 0).astype(np.int8)

# Free the memory used by the intermediate results.
del keyword_counts, cue_count, negated_count, has_alert_header

# Display the spoiler rate of the reviews with and without a keyword.
df.groupby('spoiler_keyword_flag')['has_spoiler'].agg(['count', 'mean'])

,count,mean
spoiler_keyword_flag,,
0,740626,0.025160
1,604091,0.117453


In [14]:
df

,book_id,review_text,has_spoiler,book_specificity_mean,book_specificity_max,log_word_count,log_sentence_count,log_avg_sentence_length,capitalized_word_density,log_third_first_pronoun_ratio,log_quote_count,log_exclamation_count,log_question_count,log_number_count,spoiler_keyword_count,spoiler_keyword_flag
0,18245960,This is a special book. It started slow for ab...,1,0.052968,1.406643,5.902633,2.995732,3.006204,0.013699,-1.029619,1.609438,0.000000,1.609438,1.386294,1.0,1
1,16981,Recommended by Don Katz. Avail for free in Dec...,0,0.005713,0.029646,2.397895,0.693147,2.397895,0.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
2,28684704,"A fun, fast paced science fiction thriller. I ...",1,0.036831,1.299416,6.144186,3.433987,2.803360,0.008602,-1.992430,2.564949,0.000000,1.386294,1.098612,1.0,1
3,27161156,Recommended reading to understand what is goin...,0,0.092450,0.986738,2.995732,0.693147,2.995732,0.052632,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
4,25884323,"I really enjoyed this book, and there is a lot...",1,0.039533,1.978659,6.598509,3.663562,3.010102,0.012278,-1.734601,2.302585,0.693147,1.386294,1.945910,3.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1344712,15745950,Can't wait for Travis' POV Travis Before Abby....,0,0.733474,3.424251,2.484907,1.098612,1.871802,0.454545,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
1344713,10861195,Had this on my to-read shelf forever. Will upd...,0,0.011827,0.067417,2.708050,1.386294,1.734601,0.000000,-1.098612,0.000000,0.693147,0.000000,0.000000,0.0,0
1344714,6131164,The last book left me wanting for more. I need...,0,0.067960,2.308004,4.672829,2.397895,2.451005,0.094340,-1.609438,0.000000,1.098612,0.000000,0.000000,1.0,1
1344715,10025305,Things are heating up in the second novel of I...,0,0.084760,2.780948,5.420535,2.639057,2.907321,0.048889,2.525729,0.000000,0.000000,1.609438,0.693147,0.0,0


In [ ]:
df.to_csv('df_book_reviews_spoiler_features.csv', index=False)